# Detection rules: exploratory analysis
How the gold-layer risk score was designed. Every candidate signal is scored against the IBM `is_laundering` labels (base rate ≈ 0.10% of transactions).

In [0]:
# Setup
from pyspark.sql import functions as F
from pyspark.sql import Window

CAT = "workspace.aml_lakehouse"
s = spark.table(f"{CAT}.silver_transactions")
p = spark.table(f"{CAT}.gold_account_profile")

is_ach     = F.col("payment_format") == "ACH"
cross_bank = F.col("from_bank") != F.col("to_bank")
same_ccy   = F.col("receiving_currency") == F.col("payment_currency")
big_amt    = F.col("amount_paid") >= 1000

total_pos = s.filter(F.col("is_laundering") == 1).count()
base_rate = total_pos / s.count()

def rate_by(df, col):
    display(df.groupBy(col).agg(
        F.count("*").alias("txns"),
        F.sum("is_laundering").alias("laundering"),
        (F.avg("is_laundering") * 100).alias("laundering_pct"),
    ).orderBy(F.desc("laundering_pct")))

def eval_txn(df, cond, name):
    flagged = df.filter(cond)
    n = flagged.count()
    tp = flagged.filter(F.col("is_laundering") == 1).count()
    print(f"{name}: flagged={n:,} | precision={tp/n:.2%} | recall={tp/total_pos:.1%} | lift={(tp/n)/base_rate:.1f}x")

## 1. Account-level behaviour
Laundering accounts send far more on average, but the medians are close (4 vs 2 transactions), so counts alone separate them weakly.

In [0]:
display(p.groupBy("is_laundering_account").agg(
    F.count("*").alias("accounts"),
    F.avg("txn_sent").alias("avg_txn_sent"),
    F.avg("distinct_receivers").alias("avg_distinct_receivers"),
    F.expr("percentile_approx(txn_sent, 0.5)").alias("median_txn_sent"),
    F.expr("percentile_approx(distinct_receivers, 0.5)").alias("median_receivers"),
    F.expr("percentile_approx(distinct_senders, 0.5)").alias("median_senders"),
))

acc_base = p.filter(F.col("is_laundering_account") == 1).count() / p.count()
fan = p.filter(F.col("distinct_receivers") >= 10)
n, tp = fan.count(), fan.filter(F.col("is_laundering_account") == 1).count()
print(f"Fan-out >= 10 receivers: flagged={n:,} | precision={tp/n:.1%} | baseline={acc_base:.1%}")

## 2. Transaction attributes
ACH holds ~87% of laundering at ~7x the base rate. Wire, Reinvestment and cross-currency transactions contain no laundering. That is a property of the synthetic generator and should not become a production rule.

In [0]:
rate_by(s, "payment_format")
rate_by(s.withColumn("cross_currency", (~same_ccy).cast("int")), "cross_currency")
rate_by(s.withColumn("cross_bank", cross_bank.cast("int")), "cross_bank")
rate_by(s.withColumn("hour", F.hour("txn_ts")), "hour")

## 3. Combining attributes and amount
ACH + cross-bank + amount ≥ 1k reaches 13x lift with 79% recall. Within ACH, the 10k–100k band has the highest laundering rate (amounts mix currencies, so indicative only). The midnight spike is driven by Reinvestment and Credit Card transactions, an artifact of the data.

In [0]:
eval_txn(s, is_ach, "ACH")
eval_txn(s, is_ach & cross_bank, "ACH + cross-bank")
eval_txn(s, is_ach & cross_bank & big_amt, "ACH + cross-bank + amount >= 1k")

ach = s.filter(is_ach).withColumn("amt_bucket",
    F.when(F.col("amount_paid") < 100, "1. <100")
     .when(F.col("amount_paid") < 1000, "2. 100-1k")
     .when(F.col("amount_paid") < 10000, "3. 1k-10k")
     .when(F.col("amount_paid") < 100000, "4. 10k-100k")
     .otherwise("5. >=100k"))
rate_by(ach, "amt_bucket")

display(s.filter(F.hour("txn_ts") == 0).groupBy("payment_format").count())

## 4. Rapid movement (rejected)
Outgoing transactions within 24h of an incoming one show 0.9x lift. Normal accounts move money just as fast, so the rule was dropped. A stricter amount-matched version is a possible refinement.

In [0]:
no_self = s.filter(F.col("from_account_id") != F.col("to_account_id"))
events = (no_self.select(F.col("to_account_id").alias("account_id"), "txn_ts", F.lit(0).alias("is_out"), "transaction_id")
          .union(no_self.select(F.col("from_account_id").alias("account_id"), "txn_ts", F.lit(1).alias("is_out"), "transaction_id")))
w = Window.partitionBy("account_id").orderBy("txn_ts", "is_out").rowsBetween(Window.unboundedPreceding, Window.currentRow)
out_events = (events
    .withColumn("last_in_ts", F.last(F.when(F.col("is_out") == 0, F.col("txn_ts")), ignorenulls=True).over(w))
    .filter(F.col("is_out") == 1)
    .withColumn("hours_since_in", (F.unix_timestamp("txn_ts") - F.unix_timestamp("last_in_ts")) / 3600)
    .select("transaction_id", "hours_since_in"))
s2 = s.join(out_events, "transaction_id", "left")

eval_txn(s2, F.col("hours_since_in") <= 24, "out within 24h of an in")
eval_txn(s2, F.col("hours_since_in") <= 6, "out within 6h of an in")
eval_txn(s2, is_ach & cross_bank & big_amt & (F.col("hours_since_in") <= 24), "ACH + cross-bank + >=1k + out within 24h")

## 5. Risk score bands
Points weighted by observed lift. Score 8 reaches 12% precision (~118x baseline); score ≥ 6 keeps 72% recall. This became the tiered alert queue in `03_gold_aggregates`.

In [0]:
display(spark.table(f"{CAT}.gold_transaction_scores").groupBy("risk_score").agg(
    F.count("*").alias("txns"),
    F.sum("is_laundering").alias("laundering"),
    (F.avg("is_laundering") * 100).alias("laundering_pct"),
).orderBy(F.desc("risk_score")))